# Retriever backbone comparison

Qualitative side-by-side of DINOv3 / ResNet-50 / SigLIP-2 / ViT as per-frame retrievers.

**Prereqs (run once):**
1. `python scripts/data_collection/build_all_backbones.py` — builds FAISS caches.
2. `python scripts/data_collection/extract_fixed_frames.py` — extracts fixed query frames.

See `docs/superpowers/specs/2026-05-25-retriever-backbone-comparison-design.md` for design context.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root

from PIL import Image
import matplotlib.pyplot as plt

from hanoi_caption.retrieval.backbones import (
    Dinov3Extractor, Resnet50Extractor, Siglip2Extractor, VitExtractor,
)
from hanoi_caption.retrieval.index import build_or_load_index
from hanoi_caption.retrieval.retrieve import make_retrieve_fn, make_topk_fn
from hanoi_caption.video_pipeline import sample_frames

KB_DIR         = Path('../data/kb_images')
CACHE_DIR      = Path('../data/cache')
FIXED_FRAMES   = Path('../tests/fixtures/retriever_frames')
TIMELINE_VIDEO = Path('../tests/videos/NhaThoLon_S_T03.MOV')
TOPK = 5
SAMPLE_FPS = 1.0


In [ ]:
EXTRACTORS = {
    'dinov3':   Dinov3Extractor(),
    'resnet50': Resnet50Extractor(),
    'siglip2':  Siglip2Extractor(),
    'vit':      VitExtractor(),
}
INDEXES = {n: build_or_load_index(e, KB_DIR, CACHE_DIR) for n, e in EXTRACTORS.items()}
for n, (idx, _) in INDEXES.items():
    print(f'{n:10s} dim={EXTRACTORS[n].dim:4d}  vectors={idx.ntotal}')


In [ ]:
# Section A helper: top-K grid
def show_topk_grid(queries, topk_by_model, cell_size=(2.0, 2.0)):
    model_names = list(topk_by_model.keys())
    k = len(next(iter(topk_by_model.values()))[0])
    n_rows = len(queries)
    n_cols = 1 + len(model_names) * k
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(cell_size[0] * n_cols, cell_size[1] * n_rows),
    )
    if n_rows == 1:
        axes = axes[None, :]
    for r, (label, qimg) in enumerate(queries):
        axes[r, 0].imshow(qimg)
        axes[r, 0].set_title(f'Q: {label}', fontsize=8)
        axes[r, 0].axis('off')
        c = 1
        for mname in model_names:
            for rank, res in enumerate(topk_by_model[mname][r]):
                ax = axes[r, c]
                ax.imshow(Image.open(res['path']))
                if rank == 0:
                    ax.set_title(f'{mname}\n{res["kb_id"]}\n{res["score"]:.2f}', fontsize=7)
                else:
                    ax.set_title(f'{res["kb_id"]}\n{res["score"]:.2f}', fontsize=7)
                ax.axis('off')
                c += 1
    plt.tight_layout()
    return fig

# Section B helper: timeline
def show_timeline(timeline_by_model, sample_fps=1.0):
    model_names = list(timeline_by_model.keys())
    all_kbs = sorted({kb for tl in timeline_by_model.values() for _, kb, _ in tl if kb})
    cmap = plt.cm.tab20
    palette = {kb: list(cmap(i % 20)) for i, kb in enumerate(all_kbs)}
    UNKNOWN = (0.85, 0.85, 0.85, 1.0)
    stride = 1.0 / sample_fps
    fig, ax = plt.subplots(figsize=(14, 1.0 * len(model_names) + 1))
    for row, mname in enumerate(model_names):
        for t, kb, score in timeline_by_model[mname]:
            color = list(palette.get(kb, UNKNOWN))
            color[3] = min(1.0, max(0.3, float(score)))
            ax.barh(row, stride, left=t, height=0.8, color=color, edgecolor='none')
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names)
    ax.set_xlabel('time (s)')
    handles = [plt.Line2D([0], [0], color=palette[k], lw=6, label=k) for k in all_kbs]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    return fig
